<a href="https://colab.research.google.com/github/1805shahab/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1805shahab/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# Unit of Analysis + Time Window
"""
**Unit of analysis**

One row represents the daily performance of one pseudonymized content item for one pseudonymized client on one report date from the `fact_content_daily_performance` table.

**Time window**

For this notebook I use the month **2026-03**, which is a mid-panel month. I avoid the final month because it is intended to remain a future evaluation period.

**Prediction goal**

My lane is **Refresh / Content Opportunity Scoring**. The objective is to rank content items according to their priority for review or refresh based only on information available at the decision time.

**Excluded**

Future observations, future performance, and client identifiers are excluded because they either introduce data leakage or are not useful predictive signals."""

'\n**Unit of analysis**\n\nOne row represents the daily performance of one pseudonymized content item for one pseudonymized client on one report date from the `fact_content_daily_performance` table.\n\n**Time window**\n\nFor this notebook I use the month **2026-03**, which is a mid-panel month. I avoid the final month because it is intended to remain a future evaluation period.\n\n**Prediction goal**\n\nMy lane is **Refresh / Content Opportunity Scoring**. The objective is to rank content items according to their priority for review or refresh based only on information available at the decision time.\n\n**Excluded**\n\nFuture observations, future performance, and client identifiers are excluded because they either introduce data leakage or are not useful predictive signals.'

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
"""# Fields

## Features

The model will use historical content performance signals such as:

- Clicks
- Impressions
- CTR
- Average position
- Engagement-related metrics

These describe the content before the refresh decision.

## Label (Proxy)

There is no direct "refresh priority" label.

Instead, the ranking target will be based on future content performance (or another future performance proxy developed later in the project).

## Context

Context fields include:

- report_date
- client_hash_id
- content_hash_id

These identify records but are not themselves prediction targets.

## Excluded

- client_hash_id (identifier only)
- Future clicks
- Future impressions
- Future CTR
- Any information created after the prediction date

These are excluded because they either identify entities or would leak future information into training."""

'# Fields\n\n## Features\n\nThe model will use historical content performance signals such as:\n\n- Clicks\n- Impressions\n- CTR\n- Average position\n- Engagement-related metrics\n\nThese describe the content before the refresh decision.\n\n## Label (Proxy)\n\nThere is no direct "refresh priority" label.\n\nInstead, the ranking target will be based on future content performance (or another future performance proxy developed later in the project).\n\n## Context\n\nContext fields include:\n\n- report_date\n- client_hash_id\n- content_hash_id\n\nThese identify records but are not themselves prediction targets.\n\n## Excluded\n\n- client_hash_id (identifier only)\n- Future clicks\n- Future impressions\n- Future CTR\n- Any information created after the prediction date\n\nThese are excluded because they either identify entities or would leak future information into training.'

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

# Verification Queries

The following checks verify the data contract for the Refresh / Content Opportunity Scoring lane using the `fact_content_daily_performance` warehouse table.

The verification confirms:

1. The grain (one row per report date, client, and content item).
2. The row count and time window.
3. Data availability and missing values.

In [ ]:
import pandas as pd

print("=" * 70)
print("VERIFICATION 1: Grain")
print("=" * 70)

grain = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
""").df()

if len(grain) == 0:
    print("✓ Grain verified.")
    print("Each row represents one content item for one client on one report date.")
else:
    display(grain.head())


print("\n" + "=" * 70)
print("VERIFICATION 2: Row Count and Date Window")
print("=" * 70)

summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(summary)


print("\n" + "=" * 70)
print("VERIFICATION 3: Availability")
print("=" * 70)

availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(availability)


print("\n" + "=" * 70)
print("Missing Values (Sample of 5,000 Rows)")
print("=" * 70)

display(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)


print("\n" + "=" * 70)
print("Feature Frame")
print("=" * 70)

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

feature_df = df[feature_columns].copy()

display(feature_df.head())

VERIFICATION 1: Grain


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Grain verified.
Each row represents one content item for one client on one report date.

VERIFICATION 2: Row Count and Date Window


,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31



VERIFICATION 3: Availability


,total_rows,gsc_available,ga4_available
0,9841378,3611061.0,413966.0



Missing Values (Sample of 5,000 Rows)


,0
ga4_data_available,5000
ga4_users,5000
ga4_engaged_sessions,5000
ga4_pageviews,5000
ga4_sessions,5000
ai_gemini,5000
ai_perplexity,5000
ai_chatgpt,5000
sessions_ai,5000
sessions_paid,5000



Feature Frame


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>


# Feature Availability

| Feature | Available at prediction time? |
|----------|-------------------------------|
| gsc_impressions | Yes. Historical Search Console impressions are already observed. |
| gsc_clicks | Yes. Historical clicks are known before making the ranking decision. |
| gsc_avg_position | Yes. Average search position is measured before prediction. |
| ga4_pageviews | Yes. Historical GA4 pageviews are available before the refresh decision. |
| ga4_sessions | Yes. Historical session counts are already observed. |

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("=" * 70)
print("LEAKAGE DEMONSTRATION")
print("=" * 70)

# ------------------------------------------------------------------
# Create a simple proxy label for demonstration only.
# A page is considered "high opportunity" if it received zero clicks.
# ------------------------------------------------------------------

demo_df = df.copy()

demo_df["label"] = (demo_df["gsc_clicks"] == 0).astype(int)

features = [
    "gsc_impressions",
    "gsc_avg_position"
]

X = demo_df[features].fillna(0)
y = demo_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

honest_score = accuracy_score(y_test, pred)

print(f"Accuracy without leakage: {honest_score:.4f}")

# ------------------------------------------------------------------
# DELIBERATE LEAKAGE
# Add the label itself as a feature.
# ------------------------------------------------------------------

X_leak = X.copy()

X_leak["leak_feature"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

leak_score = accuracy_score(y_test, pred)

print(f"Accuracy WITH leakage: {leak_score:.4f}")

print("\nLeakage increased the score because the model was given the answer through a label-derived feature.")

# Remove the leaking feature to restore an honest feature set
X_leak.drop(columns=["leak_feature"], inplace=True)

print("\nLeak feature removed.")

LEAKAGE DEMONSTRATION
Accuracy without leakage: 0.9120
Accuracy WITH leakage: 1.0000

Leakage increased the score because the model was given the answer through a label-derived feature.

Leak feature removed.


## Leakage Result

The second model intentionally included a feature derived directly from the target label. As expected, the evaluation score increased substantially because the model had access to information that would not exist at prediction time.

This demonstrates **label leakage**. Such features must be removed before training a real refresh-ranking model. Only historical observations that are available at the decision moment should be used.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
"""# Data Limits

The warehouse provides historical observations that support ranking and decision-making, but it also has important limitations.

- **Unbalanced history:** Different clients entered the platform at different times, so some content items have much longer histories than others.
- **Incomplete source coverage:** Some clients have only Google Search Console (GSC) data, only GA4 data, or neither for parts of the timeline, so feature availability is not uniform across all rows.
- **Observational data only:** The data shows what happened, but it cannot determine why performance changed or prove that refreshing content caused an improvement.
- **Window overlap:** Features computed from rolling time windows may overlap with one another, so they are not completely independent.
- **No true refresh label:** The warehouse does not contain a direct "content refresh priority" label, so this project relies on proxy labels or future performance signals for training.
- **Decision-support, not certainty:** The model can prioritize pages for review, but the final refresh decision should still involve human judgment and business context."""

'# Data Limits\n\nThe warehouse provides historical observations that support ranking and decision-making, but it also has important limitations.\n\n- **Unbalanced history:** Different clients entered the platform at different times, so some content items have much longer histories than others.\n- **Incomplete source coverage:** Some clients have only Google Search Console (GSC) data, only GA4 data, or neither for parts of the timeline, so feature availability is not uniform across all rows.\n- **Observational data only:** The data shows what happened, but it cannot determine why performance changed or prove that refreshing content caused an improvement.\n- **Window overlap:** Features computed from rolling time windows may overlap with one another, so they are not completely independent.\n- **No true refresh label:** The warehouse does not contain a direct "content refresh priority" label, so this project relies on proxy labels or future performance signals for training.\n- **Decision

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.